In [9]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from scripts.Loading_Dataset import LiverDataset
from tqdm import tqdm

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [11]:
#Data augmentation and normalization for encoding
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [12]:
#Load dataset
root_dir = "./Dataset" 

dataset = LiverDataset(root_dir=root_dir, transform=transform)
dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [13]:
#Load Resnet 18 Encoder (pretrained on ImageNet)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# REMOVE CLASSIFIER HEAD
model.fc = nn.Identity()

model = model.to(device)
model.eval()


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:
all_features = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(dataloader, desc="Extracting features"):
        images = images.to(device)

        # forward pass → 512-d embeddings
        features = model(images)

        # flatten just in case (already [B, 512])
        features = features.view(features.size(0), -1)

        all_features.append(features.cpu())
        all_labels.append(labels)

# combine all batches
all_features = torch.cat(all_features, dim=0)
all_labels = torch.cat(all_labels, dim=0)

Extracting features:   0%|          | 0/112 [00:00<?, ?it/s]

In [ ]:
#Output
all_features = []
all_labels = []

with torch.no_grad():
    for images, labels in dataloader:
        images = images.to(device)

        # forward pass → 512-d embeddings
        features = model(images)

        # flatten just in case (already [B, 512])
        features = features.view(features.size(0), -1)

        all_features.append(features.cpu())
        all_labels.append(labels)

# combine all batches
all_features = torch.cat(all_features, dim=0)
all_labels = torch.cat(all_labels, dim=0)


Epoch 1/1: 100%|██████████| 112/112 [14:17<00:00,  7.65s/it, acc=73.1, loss=0.654]


Epoch [1/1] Loss: 0.5252, Accuracy: 73.10%
